## Gold Layer Generator

In [0]:
# Paths
silver_jobs_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/jobs/"
silver_job_skills_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/job_skills/"
silver_quality_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/silver/data_quality_jobs/"

gold_fact_jobs_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/fact_jobs/"
gold_dim_skills_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/dim_skills/"
gold_fact_job_skills_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/fact_job_skills/"
gold_agg_top_skills_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/agg_top_skills/"
gold_agg_salary_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/agg_salary/"
gold_agg_location_path = "abfss://<CONTAINER>@<STORAGE_ACCOUNT>.dfs.core.windows.net/gold_delta/agg_location/"

In [0]:
# Reading Silver Layer
from pyspark.sql import functions as F

df_silver_jobs = spark.read.parquet(silver_jobs_path)
df_silver_job_skills = spark.read.parquet(silver_job_skills_path)
df_silver_quality = spark.read.parquet(silver_quality_path)

display(df_silver_jobs)
display(df_silver_job_skills)
display(df_silver_quality)

In [0]:
fact_jobs = df_silver_jobs.select(
    "job_id",
    "job_title",
    "company",
    "location_normalized",
    "salary_min",
    "salary_max",
    "salary_currency",
    "salary_period",
    "seniority",
    "job_family",
    "source_file",
    "ingestion_timestamp"
)

fact_jobs.write.format("delta").mode("overwrite").save(gold_fact_jobs_path)

In [0]:
# Create dim_skills
dim_skills = (
    df_silver_job_skills
    .select("skill")
    .distinct()
    .withColumn("skill_id", F.sha2(F.col("skill"), 256))
    .select("skill_id", "skill")
)

dim_skills.write.format("delta").mode("overwrite").save(gold_dim_skills_path)

In [0]:
# Create fact_job_skills
fact_job_skills = (
    df_silver_job_skills.alias("f")
    .join(dim_skills.alias("d"), on="skill", how="left")
    .select("f.job_id", "d.skill_id")
)

fact_job_skills.write.format("delta").mode("overwrite").save(gold_fact_job_skills_path)

In [0]:
# Aggregation top skills
agg_top_skills = (
    df_silver_job_skills
    .groupBy("skill")
    .agg(F.countDistinct("job_id").alias("job_count"))
    .orderBy(F.desc("job_count"))
)

agg_top_skills.write.format("delta").mode("overwrite").save(gold_agg_top_skills_path)

In [0]:
# Aggregation salary
agg_salary = (
    df_silver_jobs
    .groupBy("job_family", "seniority")
    .agg(
        F.countDistinct("job_id").alias("job_count"),
        F.avg("salary_min").alias("avg_salary_min"),
        F.avg("salary_max").alias("avg_salary_max"),
        F.min("salary_min").alias("min_salary"),
        F.max("salary_max").alias("max_salary")
    )
    .orderBy("job_family", "seniority")
)

agg_salary.write.format("delta").mode("overwrite").save(gold_agg_salary_path)

In [0]:
# Aggregation Location
agg_location = (
    df_silver_jobs
    .groupBy("location_normalized")
    .agg(F.countDistinct("job_id").alias("job_count"))
    .orderBy(F.desc("job_count"))
)

agg_location.write.format("delta").mode("overwrite").save(gold_agg_location_path)

## Check GOLD Layer

### Fact Jobs

In [0]:
df_fact_jobs = spark.read.format("delta").load(gold_fact_jobs_path)
display(df_fact_jobs)

### Dim Skills

In [0]:
df_dim_skills = spark.read.format("delta").load(gold_dim_skills_path)
display(df_dim_skills)

### Job Skills

In [0]:
df_fact_job_skills = spark.read.format("delta").load(gold_fact_job_skills_path)
display(df_fact_job_skills)

### Aggregation Skills

In [0]:
df_agg_top_skills = spark.read.format("delta").load(gold_agg_top_skills_path)
display(df_agg_top_skills)

### Aggregation Salary

In [0]:
df_agg_salary = spark.read.format("delta").load(gold_agg_salary_path)
display(df_agg_salary)

### Aggregation Location

In [0]:
df_agg_location = spark.read.format("delta").load(gold_agg_location_path)
display(df_agg_location)